# Data Exploration and Validation Steps

In this section, I load both datasets, inspect their structure, check possible matching identifiers, and validate how patients can be linked between the All Patients dataset and the eConsult dataset. These steps are important because Encounter_ID did not match across files, so patient name formatting needed to be standardized before creating the eConsult indicator.

## Step 1 — Load both files and check columns

In [1]:
import pandas as pd

# Load files
all_patients = pd.read_excel("All Patients.xlsx")
eConsult = pd.read_csv("econsults model data.csv")

# Check shape
print("All Patients shape:", all_patients.shape)
print("eConsult shape:", eConsult.shape)

# Check columns
print("\nAll Patients columns:")
print(all_patients.columns)

print("\neConsult columns:")
print(eConsult.columns)

# Preview data
display(all_patients.head())
display(eConsult.head())

/Users/samisa/Documents/Python/SAMI/.venv/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


All Patients shape: (91716, 13)
eConsult shape: (29999, 12)

All Patients columns:
Index(['Patient_Name', 'Encounter_ID', 'PCP_Name', 'Practice_Name',
       'Service_Date', 'Paid_Date', 'Paid_Amount',
       'Specialty Combined + Other', 'Diag Code', 'Coverage_Type', 'Drug_Name',
       'Transaction_Classification', 'Transaction_Desc'],
      dtype='str')

eConsult columns:
Index(['Patient First Name', 'Patient Last Name', 'Practice_Name',
       'Encounter_ID', 'Service_Date', 'Sum of Paid_Amount', 'Paid_Date',
       'Primary_Diag_Code', 'Transaction_Desc', 'Transaction_Classification',
       'Drug_Name', 'Coverage_Type'],
      dtype='str')


,Patient_Name,Encounter_ID,PCP_Name,Practice_Name,Service_Date,Paid_Date,Paid_Amount,Specialty Combined + Other,Diag Code,Coverage_Type,Drug_Name,Transaction_Classification,Transaction_Desc
0,"ABDALLA, HAGA",17361719.0,"JONES, CYNTHIA",MOSAIC HEALTH,2026-02-21,2026-03-20,743.39,Hematology,D69.6,PART A,NaN,INSTITUTIONAL,ER
1,"ABDALLA, HAGA",17366692.0,"JONES, CYNTHIA",MOSAIC HEALTH,2026-02-12,2026-02-27,300.99,Hematology,D69.6,PART B,NaN,PROFESSIONAL,SPECIALIST VISIT
2,"ABDALLA, HAGA",17371705.0,"JONES, CYNTHIA",MOSAIC HEALTH,2026-03-09,2026-03-27,17.31,Hematology,D69.6,PART B,NaN,PROFESSIONAL,LAB
3,"ABDALLA, HAGA",17379964.0,"JONES, CYNTHIA",MOSAIC HEALTH,2026-02-27,2026-03-13,19.08,Hematology,D69.6,PART B,NaN,PROFESSIONAL,LAB
4,"ABDALLA, HAGA",17385613.0,"JONES, CYNTHIA",MOSAIC HEALTH,2026-01-22,2026-02-20,66.14,Hematology,D69.6,PART B,NaN,PROFESSIONAL,OTHER PART B


,Patient First Name,Patient Last Name,Practice_Name,Encounter_ID,Service_Date,Sum of Paid_Amount,Paid_Date,Primary_Diag_Code,Transaction_Desc,Transaction_Classification,Drug_Name,Coverage_Type
0,AALIYAH,ODONNELL,COMMUNITY HEALTH CENTER OF THE NORTH COUNTRY,17368212,2026-01-05 00:00:00,$131.16,2026-01-30 00:00:00,Z00.01,PRIMARY CARE VISIT,PROFESSIONAL,NaN,PART B
1,AALIYAH,ODONNELL,COMMUNITY HEALTH CENTER OF THE NORTH COUNTRY,17421654,2026-02-26 00:00:00,$131.16,2026-03-27 00:00:00,N91.2,PRIMARY CARE VISIT,PROFESSIONAL,NaN,PART B
2,AALIYAH,ODONNELL,COMMUNITY HEALTH CENTER OF THE NORTH COUNTRY,17431780,2026-03-05 00:00:00,$167.42,2026-03-14 00:00:00,NaN,DENTAL,PROFESSIONAL,NaN,PART B
3,AALIYAH,ODONNELL,COMMUNITY HEALTH CENTER OF THE NORTH COUNTRY,17436862,2026-02-02 00:00:00,$106.00,2026-02-27 00:00:00,H66.91,OTHER PART B,PROFESSIONAL,NaN,PART B
4,AALIYAH,ODONNELL,COMMUNITY HEALTH CENTER OF THE NORTH COUNTRY,17447597,2026-02-02 00:00:00,$0.00,2026-02-02 00:00:00,NaN,PART D RX,PHARMACY,AMOXICILLIN,PART D


## Step 2 — Check if Encounter_ID matches

In [2]:
# Convert Encounter_ID to string in both datasets
all_patients['Encounter_ID'] = all_patients['Encounter_ID'].astype(str)
eConsult['Encounter_ID'] = eConsult['Encounter_ID'].astype(str)

# Check how many Encounter_IDs match
matches = all_patients['Encounter_ID'].isin(eConsult['Encounter_ID']).sum()

print("Matching Encounter_IDs:", matches)

print("Unique Encounter_IDs in All Patients:", all_patients['Encounter_ID'].nunique())
print("Unique Encounter_IDs in eConsult:", eConsult['Encounter_ID'].nunique())

Matching Encounter_IDs: 0
Unique Encounter_IDs in All Patients: 47183
Unique Encounter_IDs in eConsult: 26163


## Step 3 — Create patient name in eConsult and check matches

In [3]:
# Create Patient_Name in eConsult
eConsult['Patient_Name'] = (
    eConsult['Patient First Name'].astype(str).str.strip().str.upper()
    + " " +
    eConsult['Patient Last Name'].astype(str).str.strip().str.upper()
)

# Clean Patient_Name in all_patients
all_patients['Patient_Name'] = (
    all_patients['Patient_Name'].astype(str).str.strip().str.upper()
)

# Clean Practice_Name
all_patients['Practice_Name'] = all_patients['Practice_Name'].astype(str).str.strip().str.upper()
eConsult['Practice_Name'] = eConsult['Practice_Name'].astype(str).str.strip().str.upper()

# Create matching key
all_patients['Match_Key'] = all_patients['Patient_Name'] + " | " + all_patients['Practice_Name']
eConsult['Match_Key'] = eConsult['Patient_Name'] + " | " + eConsult['Practice_Name']

# Check matches
matches = all_patients['Match_Key'].isin(eConsult['Match_Key']).sum()

print("Matching Patient + Practice rows:", matches)
print("Unique Match_Key in All Patients:", all_patients['Match_Key'].nunique())
print("Unique Match_Key in eConsult:", eConsult['Match_Key'].nunique())

Matching Patient + Practice rows: 0
Unique Match_Key in All Patients: 543
Unique Match_Key in eConsult: 264


## Step 4 — Check name formats

In [4]:
print("All Patients sample names:")
print(all_patients['Patient_Name'].dropna().unique()[:20])

print("\neConsult sample names:")
print(eConsult['Patient_Name'].dropna().unique()[:20])

print("\nAll Patients sample practices:")
print(all_patients['Practice_Name'].dropna().unique()[:10])

print("\neConsult sample practices:")
print(eConsult['Practice_Name'].dropna().unique()[:10])

All Patients sample names:
<ArrowStringArray>
[          'ABDALLA, HAGA', 'ABDULAHAMIN, YASMINTARA',
          'ACHARYA, TANKA',          'ACKERMAN, LISA',
        'ACOSTA, SAMANTHA',           'AIKENS, DEVEN',
           'AKTER, MAHIYA',         'ALEXANDER, CODY',
        'ALHOULAL, KHALIL',              'ALI, WALAA',
    'AL-KHUZAIE, NAREEMAN',          'ALLEN, BRADLEY',
      'ALLINGTON, MAVERIK',        'ALMALETTI, SHERI',
       'ALVARADO, RICHARD',    'AMADOR LOPEZ, MELVIN',
           'AMES, BRANDON',        'AMIDON, MADELYNN',
       'APPLEGATE, AUSTIN',      'ARGUIJO, ALEXANDER']
Length: 20, dtype: str

eConsult sample names:
<ArrowStringArray>
[   'AALIYAH ODONNELL',     'ABIGAIL CARMODY',         'AIRYS CYRUS',
      'BLAKE MATTHEWS',        'EMILLY STONE',      'EMILY TROMBLEY',
        'GEYAH GAGNON',        'IZEYAH DUPEE',      'JAYLA ODONNELL',
     'LORI BRONCHETTI', 'SAMANTHA COUNTRYMAN',        'SHAWN FLEURY',
       'SKYLER CROCIE',         'TINA AUBREY',        'VER

## Step 5 — Fix name format and match again

In [5]:
# Convert all_patients name from "LAST, FIRST" to "FIRST LAST"
name_split = all_patients['Patient_Name'].str.split(',', expand=True)

all_patients['Last_Name'] = name_split[0].str.strip()
all_patients['First_Name'] = name_split[1].str.strip()

all_patients['Patient_Name_Clean'] = (
    all_patients['First_Name'] + " " + all_patients['Last_Name']
)

# eConsult already has FIRST LAST format
eConsult['Patient_Name_Clean'] = eConsult['Patient_Name']

# Create match key using patient name only first
all_patients['Name_Key'] = all_patients['Patient_Name_Clean'].str.upper().str.strip()
eConsult['Name_Key'] = eConsult['Patient_Name_Clean'].str.upper().str.strip()

# Check name matches
matches = all_patients['Name_Key'].isin(eConsult['Name_Key']).sum()

print("Matching patient name rows:", matches)

print("Unique patient names in All Patients:", all_patients['Name_Key'].nunique())
print("Unique patient names in eConsult:", eConsult['Name_Key'].nunique())

Matching patient name rows: 59350
Unique patient names in All Patients: 469
Unique patient names in eConsult: 262


## Step 6 — Create the prediction target variable

In [6]:
# Create target variable
# 1 = patient received eConsult
# 0 = patient did not receive eConsult

all_patients['Received_eConsult'] = all_patients['Name_Key'].isin(
    eConsult['Name_Key']
).astype(int)

# Check result
print(all_patients['Received_eConsult'].value_counts())

print("\nPercentage:")
print(all_patients['Received_eConsult'].value_counts(normalize=True) * 100)

Received_eConsult
1    59350
0    32366
Name: count, dtype: int64

Percentage:
Received_eConsult
1    64.710628
0    35.289372
Name: proportion, dtype: float64


* 59,350 encounter rows belong to patients who appeared in the eConsult dataset
* 32,366 encounter rows belong to patients who did not appear in the eConsult dataset

## Step 7 — Clean cost/date columns

In [7]:
# Clean Paid_Amount
all_patients['Paid_Amount'] = (
    all_patients['Paid_Amount']
    .astype(str)
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
)

all_patients['Paid_Amount'] = pd.to_numeric(
    all_patients['Paid_Amount'],
    errors='coerce'
)

# Convert dates
all_patients['Service_Date'] = pd.to_datetime(
    all_patients['Service_Date'],
    errors='coerce'
)

all_patients['Paid_Date'] = pd.to_datetime(
    all_patients['Paid_Date'],
    errors='coerce'
)

# Check
print(all_patients[['Paid_Amount', 'Service_Date', 'Paid_Date']].head())
print(all_patients[['Paid_Amount', 'Service_Date', 'Paid_Date']].isna().sum())

   Paid_Amount Service_Date  Paid_Date
0       743.39   2026-02-21 2026-03-20
1       300.99   2026-02-12 2026-02-27
2        17.31   2026-03-09 2026-03-27
3        19.08   2026-02-27 2026-03-13
4        66.14   2026-01-22 2026-02-20
Paid_Amount     2
Service_Date    3
Paid_Date       3
dtype: int64


# Research Question 1

The goal of this project is to evaluate the effectiveness of the eConsult telehealth program and determine whether the service should continue based on its impact on healthcare utilization and costs.

Main Research Question:
- Does participation in the eConsult program reduce healthcare utilization and healthcare costs?

## Step 1 — Find each patient’s first eConsult date

In [8]:
# Make sure eConsult Service_Date is datetime
eConsult['Service_Date'] = pd.to_datetime(
    eConsult['Service_Date'],
    errors='coerce'
)

# Find first eConsult date for each patient
first_econsult_date = (
    eConsult
    .groupby('Name_Key', as_index=False)['Service_Date']
    .min()
    .rename(columns={'Service_Date': 'First_eConsult_Date'})
)

# Check result
print("Number of eConsult patients:", first_econsult_date.shape[0])
display(first_econsult_date.head())

Number of eConsult patients: 262


,Name_Key,First_eConsult_Date
0,AALIYAH ODONNELL,2024-01-25
1,AARON WOOD,2024-02-13
2,ABBYLYNN BUTLER,2024-02-14
3,ABIGAIL CARMODY,2024-01-11
4,ACHSEL EATON,2024-01-02


## Step 2 — Merge first eConsult date into all patients data

In [9]:
# Merge first eConsult date into all_patients
all_patients = all_patients.merge(
    first_econsult_date,
    on='Name_Key',
    how='left'
)

# Preview
display(
    all_patients[
        [
            'Name_Key',
            'Service_Date',
            'First_eConsult_Date'
        ]
    ].head()
)

# Check how many patients received eConsult
print(
    all_patients['First_eConsult_Date']
    .notna()
    .sum()
)

,Name_Key,Service_Date,First_eConsult_Date
0,HAGA ABDALLA,2026-02-21,NaT
1,HAGA ABDALLA,2026-02-12,NaT
2,HAGA ABDALLA,2026-03-09,NaT
3,HAGA ABDALLA,2026-02-27,NaT
4,HAGA ABDALLA,2026-01-22,NaT


59350


## Step 3 — Create Before/After Periods


In [10]:
# Create period column
all_patients['Period'] = 'No_eConsult'

# Before eConsult
all_patients.loc[
    all_patients['Service_Date'] < all_patients['First_eConsult_Date'],
    'Period'
] = 'Before'

# After eConsult
all_patients.loc[
    all_patients['Service_Date'] >= all_patients['First_eConsult_Date'],
    'Period'
] = 'After'

# Check counts
print(all_patients['Period'].value_counts())

# Preview
display(
    all_patients[
        [
            'Name_Key',
            'Service_Date',
            'First_eConsult_Date',
            'Period'
        ]
    ].head(10)
)

Period
After          59347
No_eConsult    32366
Before             3
Name: count, dtype: int64


,Name_Key,Service_Date,First_eConsult_Date,Period
0,HAGA ABDALLA,2026-02-21,NaT,No_eConsult
1,HAGA ABDALLA,2026-02-12,NaT,No_eConsult
2,HAGA ABDALLA,2026-03-09,NaT,No_eConsult
3,HAGA ABDALLA,2026-02-27,NaT,No_eConsult
4,HAGA ABDALLA,2026-01-22,NaT,No_eConsult
5,HAGA ABDALLA,2026-02-21,NaT,No_eConsult
6,HAGA ABDALLA,2026-02-27,NaT,No_eConsult
7,HAGA ABDALLA,2026-02-27,NaT,No_eConsult
8,HAGA ABDALLA,2026-01-22,NaT,No_eConsult
9,HAGA ABDALLA,2026-02-28,NaT,No_eConsult


## RESULT:
The before-and-after analysis revealed an important limitation in the current datasets. Only 3 encounters were identified before patients’ first eConsult dates, while almost all encounters were classified as occurring after eConsult participation. This suggests that the All Patients claims dataset primarily contains records from a later time period than the eConsult dataset. For example, many eConsult dates occur in 2024, while much of the claims data occurs in 2026. As a result, there is very limited historical “before” data available, making a true pre/post utilization analysis not feasible with the current files alone. However, this is still an important finding because it identifies a data coverage limitation and helps guide the next phase of the analysis.

# Research Question 2
Is participation in the eConsult telehealth program associated with reduced unmanaged healthcare utilization and costs?”

## Step 1 — Create unmanaged encounter flag

In [11]:
# Define unmanaged encounter types
unmanaged_types = ['ER', 'INPATIENT', 'OUTPATIENT']

# Create unmanaged encounter flag
all_patients['Unmanaged_Flag'] = (
    all_patients['Transaction_Desc']
    .isin(unmanaged_types)
    .astype(int)
)

# Create unmanaged cost column
all_patients['Unmanaged_Cost'] = (
    all_patients['Paid_Amount'] *
    all_patients['Unmanaged_Flag']
)

# Check results
display(
    all_patients[
        [
            'Transaction_Desc',
            'Paid_Amount',
            'Unmanaged_Flag',
            'Unmanaged_Cost'
        ]
    ].head(20)
)

# Summary statistics
print("Total unmanaged encounters:")
print(all_patients['Unmanaged_Flag'].sum())

print("\nTotal unmanaged cost:")
print(all_patients['Unmanaged_Cost'].sum())

,Transaction_Desc,Paid_Amount,Unmanaged_Flag,Unmanaged_Cost
0,ER,743.39,1,743.39
1,SPECIALIST VISIT,300.99,0,0.00
2,LAB,17.31,0,0.00
3,LAB,19.08,0,0.00
4,OTHER PART B,66.14,0,0.00
5,OTHER PART B,148.35,0,0.00
6,OTHER PART B,65.98,0,0.00
7,LAB,0.81,0,0.00
8,PART D RX,0.00,0,0.00
9,PART D RX,0.00,0,0.00


Total unmanaged encounters:
15985

Total unmanaged cost:
6110527.07


## Step 2 — Filter data to match the Power BI date range

In [12]:
# Filter data to match BI date range
filtered_data = all_patients[
    (all_patients['Service_Date'] >= '2024-11-01') &
    (all_patients['Service_Date'] <= '2026-01-01')
].copy()

# Check shape
print("Filtered dataset shape:", filtered_data.shape)

# Check date range
print("\nMinimum date:", filtered_data['Service_Date'].min())
print("Maximum date:", filtered_data['Service_Date'].max())

Filtered dataset shape: (51080, 23)

Minimum date: 2024-11-01 00:00:00
Maximum date: 2026-01-01 00:00:00


## Step 3 — Recalculate unmanaged encounters and costs

In [13]:
# Define unmanaged encounter types
unmanaged_types = ['ER', 'INPATIENT', 'OUTPATIENT']

# Create unmanaged flag
filtered_data['Unmanaged_Flag'] = (
    filtered_data['Transaction_Desc']
    .isin(unmanaged_types)
    .astype(int)
)

# Create unmanaged cost
filtered_data['Unmanaged_Cost'] = (
    filtered_data['Paid_Amount'] *
    filtered_data['Unmanaged_Flag']
)

# Summary
print("Total unmanaged encounters:")
print(filtered_data['Unmanaged_Flag'].sum())

print("\nTotal unmanaged cost:")
print(filtered_data['Unmanaged_Cost'].sum())

Total unmanaged encounters:
8934

Total unmanaged cost:
3660468.6399999997


## Step 4 — Compare unmanaged utilization by eConsult participation

In [14]:
# Compare eConsult vs non-eConsult groups
comparison = filtered_data.groupby('Received_eConsult').agg({
    'Unmanaged_Flag': 'sum',
    'Unmanaged_Cost': 'sum',
    'Paid_Amount': 'sum',
    'Encounter_ID': 'count'
}).reset_index()

# Rename columns
comparison.columns = [
    'Received_eConsult',
    'Total_Unmanaged_Encounters',
    'Total_Unmanaged_Cost',
    'Total_Cost',
    'Total_Encounters'
]

# Average unmanaged cost per encounter
comparison['Avg_Unmanaged_Cost_Per_Encounter'] = (
    comparison['Total_Unmanaged_Cost'] /
    comparison['Total_Encounters']
)

# Average total cost per encounter
comparison['Avg_Total_Cost_Per_Encounter'] = (
    comparison['Total_Cost'] /
    comparison['Total_Encounters']
)

# Display
display(comparison)

,Received_eConsult,Total_Unmanaged_Encounters,Total_Unmanaged_Cost,Total_Cost,Total_Encounters,Avg_Unmanaged_Cost_Per_Encounter,Avg_Total_Cost_Per_Encounter
0,0,2126,1103135.59,2030080.93,17844,61.821093,113.768266
1,1,6808,2557333.05,4256906.17,33236,76.944670,128.081182


## RESULT:
eConsult patients demonstrated higher unmanaged healthcare utilization and costs compared to non-eConsult patients during the study period. This may indicate that the eConsult program is being utilized by higher-risk or more medically complex patients who require greater healthcare services.

# Research Question 3
Who are the highest-cost eConsult patients?

## Step 1 — Keep only eConsult patients

In [15]:
# Keep only eConsult patients
econsult_patients = filtered_data[
    filtered_data['Received_eConsult'] == 1
].copy()

# Check shape
print("eConsult dataset shape:", econsult_patients.shape)

# Preview
display(
    econsult_patients[
        [
            'Name_Key',
            'Paid_Amount',
            'Transaction_Desc',
            'Coverage_Type'
        ]
    ].head()
)

eConsult dataset shape: (33236, 23)


,Name_Key,Paid_Amount,Transaction_Desc,Coverage_Type
333,LISA ACKERMAN,132.59,OUTPATIENT,PART A
334,LISA ACKERMAN,141.20,OUTPATIENT,PART A
335,LISA ACKERMAN,138.76,OUTPATIENT,PART A
336,LISA ACKERMAN,141.49,OUTPATIENT,PART A
337,LISA ACKERMAN,165.53,OUTPATIENT,PART A


In [16]:
# Clean eConsult paid amount column
eConsult['Sum of Paid_Amount'] = (
    eConsult['Sum of Paid_Amount']
    .astype(str)
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
)

eConsult['Sum of Paid_Amount'] = pd.to_numeric(
    eConsult['Sum of Paid_Amount'],
    errors='coerce'
)

# Aggregate patient-level metrics
patient_costs = eConsult.groupby('Name_Key').agg({
    'Sum of Paid_Amount': 'sum',
    'Encounter_ID': 'count'
}).reset_index()

# Rename columns
patient_costs.columns = [
    'Patient',
    'Total_eConsult_Cost',
    'Total_eConsult_Encounters'
]

# Sort highest cost patients
patient_costs = patient_costs.sort_values(
    by='Total_eConsult_Cost',
    ascending=False
)

# Top 10 patients
display(patient_costs.head(10))

,Patient,Total_eConsult_Cost,Total_eConsult_Encounters
24,BAILEY WENDEL,208610.74,251
72,DOUGLAS NOLAN,179894.61,326
207,NANCY ROOT,174331.94,767
109,JANINE WORD,156509.43,274
44,CHARNIE NORTON,79196.18,701
87,GENEVEE OLIVER,79128.64,178
259,WALTER MERRILL,77383.32,340
185,MARIA CHAMBERLAIN,77264.85,152
222,RICHARD ALVARADO,74049.16,1125
102,JACK STEVENS,63829.63,203


In [17]:
# Average cost per encounter
patient_costs['Avg_Cost_Per_Encounter'] = (
    patient_costs['Total_eConsult_Cost'] /
    patient_costs['Total_eConsult_Encounters']
)

# Sort by average cost
avg_cost_patients = patient_costs.sort_values(
    by='Avg_Cost_Per_Encounter',
    ascending=False
)

# Display top 10
display(avg_cost_patients.head(10))

,Patient,Total_eConsult_Cost,Total_eConsult_Encounters,Avg_Cost_Per_Encounter
24,BAILEY WENDEL,208610.74,251,831.118486
66,DENNIS MALLETT,16802.51,24,700.104583
109,JANINE WORD,156509.43,274,571.202299
72,DOUGLAS NOLAN,179894.61,326,551.823957
185,MARIA CHAMBERLAIN,77264.85,152,508.321382
87,GENEVEE OLIVER,79128.64,178,444.542921
214,PAXTON RILEY,17278.98,42,411.404286
88,GEYAH GAGNON,48214.10,135,357.141481
225,ROGER MEJEAN,36244.79,103,351.891165
102,JACK STEVENS,63829.63,203,314.431675
